# Ranking inicial (antes de Evo2)

**Responsable:** Integrante 1
**Proyecto:** Caso 1 · PAH (GRCh38, NM_000277.3, NP_000268.1, PDB 6HYC)

Este notebook construye el **Top 5 inicial** usando solo evidencia previa a los modelos fundacionales: localización, consecuencia molecular, frecuencia poblacional y naturaleza del cambio proteico. Después agrega la comparación con Evo2 y prepara la tabla de entrada para RF3.

Entradas:
- `resultados/01_anotaciones_genomicas.tsv`
- `resultados/anotacion_transcrito_proteina.tsv`
- `resultados/propuesta_top5_integrante3.tsv`
- `resultados/evo2_deltalogp.tsv` (solo en la sección 6)

Salidas:
- `resultados/ranking_inicial.tsv`
- `resultados/PAH_pre_vs_Evo2.tsv` (entrada para `resultados/rf3/Entradas_RF3.ipynb`)
- `resultados/tabla_maestra.tsv` (columnas hasta Evo2)


In [ ]:
# Configuración: correr esta celda primero
import os
if not os.path.exists("database"):          # en Colab: clonar el repositorio
    if not os.path.exists("Caso_1_PHA_Humano"):
        !git clone https://github.com/XomiDessu/Caso_1_PHA_Humano.git
    %cd Caso_1_PHA_Humano

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 1. Cargar las anotaciones del equipo

Se unen la anotación genómica (región, frecuencia, distancia al borde de exón) y la anotación de transcrito y proteína (cambio c., cambio p., consecuencia, BLOSUM62) por ID de variante.


In [1]:
gen = pd.read_csv("resultados/01_anotaciones_genomicas.tsv", sep="\t")
prot = pd.read_csv("resultados/anotacion_transcrito_proteina.tsv", sep="\t")

gen = gen.rename(columns={"variant_id": "ID"})
cols_prot = ["ID", "transcript_change", "protein_change", "codon_number",
             "aa_WT", "aa_ALT", "consequence", "delta_aa", "BLOSUM62"]
df = gen.merge(prot[cols_prot], on="ID", how="left")

assert len(df) == 15, "Deben ser 15 variantes"
print(df[["ID", "genomic_location", "variant_type", "ubicacion", "distancia_borde_exon",
          "POPAF", "transcript_change", "protein_change", "consequence", "BLOSUM62"]].to_string(index=False))


    ID          genomic_location variant_type ubicacion  distancia_borde_exon    POPAF transcript_change     protein_change       consequence  BLOSUM62
VAR001           chr12:102840437          SNV       CDS                    37 0.041240         c.1278T>C          p.Asn426=          sinonima       NaN
VAR002           chr12:102840473          SNV       CDS                    42 0.007790         c.1242C>T          p.Tyr414=          sinonima       NaN
VAR003           chr12:102840493          SNV       CDS                    22 0.000920         c.1222C>T        p.Arg408Trp          missense      -3.0
VAR004           chr12:102843690          SNV       CDS                    44 0.849530         c.1155C>G          p.Leu385=          sinonima       NaN
VAR005           chr12:102844388          SNV       CDS                    43 0.000004         c.1013A>C        p.Asp338Ala          missense      -2.0
VAR006           chr12:102846901          SNV       CDS                     6 0.015820  

## 2. Criterios de priorización

Cada variante recibe puntos en cinco criterios. Los puntos se suman y se ordena de mayor a menor. Los criterios usan solo información disponible **antes** de Evo2.

| Criterio | Regla | Puntos |
|-|-|-|
| Consecuencia molecular | missense o deleción in-frame | 3 |
| | sinónima | 1 |
| | intrón o fuera del CDS | 0.5 |
| Cercanía a splicing | a 3 nt o menos del borde de exón | +1 |
| Frecuencia (POPAF) | < 0.0001 (ultra-rara o ausente) | 2 |
| | 0.0001 a < 0.001 | 1.5 |
| | 0.001 a < 0.01 | 1 |
| | 0.01 a < 0.05 | 0.5 |
| | ≥ 0.05, o REF minoritaria | 0 |
| Cambio químico (BLOSUM62) | −3 o menor / −2 / −1 / 0 / positivo | 2 / 1.5 / 1 / 0.5 / 0 |
| | deleción in-frame de 3 aa | 2 |
| | +0.5 si cambia la carga o involucra Gly, Pro o Cys | +0.5 |
| Dominio | catalítico (aa 118–427) | 1 |
| | regulador (1–117) o tetramerización (428–452) | 0.5 |

Desempates: primero el BLOSUM62 más negativo y después la frecuencia más baja.

Límites de dominio: se usa la división habitual de PAH en tres dominios (regulador 1–117, catalítico 118–427, tetramerización 428–452). UniProt P00439 confirma el dominio ACT regulador en los residuos 36–114 y los residuos que unen hierro en el sitio activo (285, 290 y 330); no da coordenadas para los dominios catalítico ni de tetramerización. VAR011 (Thr117) cae justo en la frontera entre el dominio regulador y el catalítico, así que su dominio depende de la convención usada.

**Advertencias del README:** una variante rara no es necesariamente dañina y una frecuente no es automáticamente irrelevante. El puntaje ordena la discusión; no demuestra patogenicidad.


In [2]:
CARGA = {"D": -1, "E": -1, "K": 1, "R": 1, "H": 0.5}
ESPECIALES = set("GPC")

def puntos_consecuencia(r):
    c = str(r["consequence"])
    if c in ("missense", "delecion_in_frame"):
        return 3
    if c == "sinonima":
        return 1
    return 0.5

def puntos_splicing(r):
    return 1 if r["distancia_borde_exon"] <= 3 else 0

def puntos_frecuencia(r):
    if r["is_major_allele"]:
        return 0
    p = r["POPAF"]
    if p < 0.0001: return 2
    if p < 0.001:  return 1.5
    if p < 0.01:   return 1
    if p < 0.05:   return 0.5
    return 0

def puntos_quimica(r):
    if r["consequence"] == "delecion_in_frame":
        return 2
    if r["consequence"] != "missense":
        return 0
    b = r["BLOSUM62"]
    base = 2 if b <= -3 else 1.5 if b == -2 else 1 if b == -1 else 0.5 if b == 0 else 0
    wt, alt = r["aa_WT"], r["aa_ALT"]
    extra = 0.5 if (CARGA.get(wt, 0) != CARGA.get(alt, 0) or wt in ESPECIALES or alt in ESPECIALES) else 0
    return base + extra

def dominio(r):
    n = r["codon_number"]
    if pd.isna(n):
        return "fuera del CDS"
    n = int(n)
    return "regulador" if n <= 117 else "catalítico" if n <= 427 else "tetramerización"

def puntos_dominio(r):
    if r["consequence"] not in ("missense", "delecion_in_frame"):
        return 0
    return {"catalítico": 1, "regulador": 0.5, "tetramerización": 0.5}.get(r["dominio"], 0)

df["dominio"] = df.apply(dominio, axis=1)
df["pts_consecuencia"] = df.apply(puntos_consecuencia, axis=1)
df["pts_splicing"] = df.apply(puntos_splicing, axis=1)
df["pts_frecuencia"] = df.apply(puntos_frecuencia, axis=1)
df["pts_quimica"] = df.apply(puntos_quimica, axis=1)
df["pts_dominio"] = df.apply(puntos_dominio, axis=1)
df["puntaje_inicial"] = df[["pts_consecuencia", "pts_splicing", "pts_frecuencia",
                            "pts_quimica", "pts_dominio"]].sum(axis=1)

df = df.sort_values(["puntaje_inicial", "BLOSUM62", "POPAF"],
                    ascending=[False, True, True], na_position="last").reset_index(drop=True)
df["rank_inicial"] = range(1, len(df) + 1)

print(df[["rank_inicial", "ID", "protein_change", "consequence", "dominio", "POPAF", "BLOSUM62",
          "pts_consecuencia", "pts_splicing", "pts_frecuencia", "pts_quimica", "pts_dominio",
          "puntaje_inicial"]].to_string(index=False))


 rank_inicial     ID     protein_change       consequence       dominio    POPAF  BLOSUM62  pts_consecuencia  pts_splicing  pts_frecuencia  pts_quimica  pts_dominio  puntaje_inicial
            1 VAR003        p.Arg408Trp          missense    catalítico 0.000920      -3.0               3.0             0             1.5          2.5          1.0              8.0
            2 VAR005        p.Asp338Ala          missense    catalítico 0.000004      -2.0               3.0             0             2.0          2.0          1.0              8.0
            3 VAR007        p.Gly247Asp          missense    catalítico 0.000010      -1.0               3.0             0             2.0          1.5          1.0              7.5
            4 VAR011        p.Thr117Ile          missense     regulador 0.000050      -1.0               3.0             1             2.0          1.0          0.5              7.5
            5 VAR012 p.Ala104_Val106del delecion_in_frame     regulador 0.000010       NaN

## 3. Comparación con la propuesta previa

Sirve para comprobar que el puntaje no contradice el razonamiento del equipo sin una razón.


In [3]:
prop3 = pd.read_csv("resultados/propuesta_top5_integrante3.tsv", sep="\t")
top5_puntaje = df.head(5)["ID"].tolist()
top5_int3 = prop3.sort_values("propuesta_rank")["ID"].tolist()

print("Top 5 por puntaje:     ", top5_puntaje)
print("Top 5 propuesta previa:", top5_int3)
print("En ambos:              ", [v for v in top5_puntaje if v in top5_int3])
print("Solo por puntaje:      ", [v for v in top5_puntaje if v not in top5_int3])
print("Solo en la propuesta:  ", [v for v in top5_int3 if v not in top5_puntaje])


Top 5 por puntaje:      ['VAR003', 'VAR005', 'VAR007', 'VAR011', 'VAR012']
Top 5 propuesta previa: ['VAR012', 'VAR003', 'VAR005', 'VAR007', 'VAR015']
En ambos:               ['VAR003', 'VAR005', 'VAR007', 'VAR012']
Solo por puntaje:       ['VAR011']
Solo en la propuesta:   ['VAR015']


## 4. Ranking inicial (Top 5)

Tabla con el formato del README: Rank, Variante, Región, Consecuencia, Evidencia y Justificación. Se guarda en `resultados/ranking_inicial.tsv` y **no se modifica después de este punto**.


In [4]:
NOMBRE_CONSEC = {"missense": "missense", "delecion_in_frame": "deleción in-frame",
                 "sinonima": "sinónima", "fuera_del_CDS": "fuera del CDS"}

def evidencia(r):
    partes = [f"POPAF {r['POPAF']:g}"]
    if r["consequence"] == "missense":
        partes.append(f"BLOSUM62 {int(r['BLOSUM62'])}")
    partes.append(f"dominio {r['dominio']}")
    partes.append(f"{int(r['distancia_borde_exon'])} nt del borde de exón")
    return "; ".join(partes)

def justificacion(r):
    if r["consequence"] == "delecion_in_frame":
        txt = "Elimina 3 aminoácidos consecutivos sin cambiar el marco de lectura"
    else:
        wt, alt = r["protein_change"][2:5], r["protein_change"][-3:]
        txt = f"Cambia {wt} por {alt} en la posición {int(r['codon_number'])}"
        if r["BLOSUM62"] < 0:
            txt += f" (BLOSUM62 {int(r['BLOSUM62'])}: sustitución poco frecuente entre proteínas homólogas)"
    txt += f"; variante {r['af_category'].split(' (')[0].lower()} en la población"
    if r["pts_splicing"]:
        txt += "; además está a 2 nt del borde de exón y podría afectar el splicing"
    if r["dominio"] == "catalítico":
        txt += "; cae en el dominio catalítico"
    return txt + "."

top5 = df.head(5).copy()
ranking_inicial = pd.DataFrame({
    "Rank": top5["rank_inicial"],
    "Variante": top5["ID"] + " " + top5["protein_change"],
    "Region": top5["ubicacion"] + " (" + top5["dominio"] + ")",
    "Consecuencia": top5["consequence"].map(NOMBRE_CONSEC),
    "Evidencia": top5.apply(evidencia, axis=1),
    "Justificacion": top5.apply(justificacion, axis=1),
    "Puntaje": top5["puntaje_inicial"],
})
ranking_inicial.to_csv("resultados/ranking_inicial.tsv", sep="\t", index=False)
print(ranking_inicial.to_string(index=False))


 Rank                  Variante           Region      Consecuencia                                                               Evidencia                                                                                                                                                                                                      Justificacion  Puntaje
    1        VAR003 p.Arg408Trp CDS (catalítico)          missense POPAF 0.00092; BLOSUM62 -3; dominio catalítico; 22 nt del borde de exón                                        Cambia Arg por Trp en la posición 408 (BLOSUM62 -3: sustitución poco frecuente entre proteínas homólogas); variante muy rara en la población; cae en el dominio catalítico.      8.0
    2        VAR005 p.Asp338Ala CDS (catalítico)          missense   POPAF 4e-06; BLOSUM62 -2; dominio catalítico; 43 nt del borde de exón                                      Cambia Asp por Ala en la posición 338 (BLOSUM62 -2: sustitución poco frecuente entre proteínas homólogas);

## 5. Cierre del ranking inicial

- **Observamos:** de las 15 variantes, 7 son missense, 1 es una deleción in-frame de 3 aminoácidos, 6 son sinónimas y 1 es intrónica. VAR004 tiene POPAF 0.85: su REF es el alelo minoritario.
- **Inferimos que:** las variantes que cambian la proteína, son raras y alteran mucho la química del residuo son las candidatas más plausibles a afectar la función de PAH.
- **La evidencia aporta:** consecuencia molecular, frecuencia poblacional, puntaje BLOSUM62 y dominio afectado.
- **Concluimos:** el Top 5 inicial queda guardado en `resultados/ranking_inicial.tsv` como primer nivel de razonamiento.
- **Falta evidencia:** ninguna de estas capas mide actividad enzimática; el efecto sobre el splicing de las sinónimas e intrónicas no se evaluó; los límites de dominio son aproximados.


## 6. Comparación con Evo2 (después del ranking inicial)

Se añade el ΔlogP de Evo2 desde `resultados/evo2_deltalogp.tsv`. Un valor más negativo indica que Evo2 favorece más el alelo de referencia en ese contexto. **No mide patogenicidad.**

Se ordena por cada una de las 4 medidas y se usa el promedio de esos 4 lugares como orden de consenso de Evo2. VAR012 es una deleción y Evo2 no la evaluó (NaN).


In [5]:
evo = pd.read_csv("resultados/evo2_deltalogp.tsv", sep="\t")
medidas = ["delta_sitio_genoma", "delta_ventana_genoma", "delta_sitio_gen", "delta_ventana_gen"]

comp = df.merge(evo, on="ID", how="left")
for m in medidas:
    comp["rank_" + m] = comp[m].rank(method="min")          # 1 = más negativo
comp["rank_evo2"] = comp[["rank_" + m for m in medidas]].mean(axis=1).rank(method="min")
comp["cambio_vs_inicial"] = comp["rank_inicial"] - comp["rank_evo2"]  # positivo = Evo2 la sube

print(comp.sort_values("rank_evo2")[["ID", "protein_change", "consequence", "rank_inicial", "rank_evo2",
                                     "cambio_vs_inicial"] + medidas].to_string(index=False))


    ID     protein_change       consequence  rank_inicial  rank_evo2  cambio_vs_inicial  delta_sitio_genoma  delta_ventana_genoma  delta_sitio_gen  delta_ventana_gen
VAR007        p.Gly247Asp          missense             3        1.0                2.0               -7.83                -22.99            -1.48             -21.58
VAR014         p.Glu76Asp          missense             7        2.0                5.0               -5.94                -10.28            -5.85              -8.24
VAR005        p.Asp338Ala          missense             2        3.0               -1.0               -4.14                -18.98            -0.20             -29.51
VAR003        p.Arg408Trp          missense             1        4.0               -3.0               -5.72                -16.06             0.58             -31.65
VAR011        p.Thr117Ile          missense             4        5.0               -1.0                0.15                -11.50            -5.18              -5.90
VAR0

### Lectura de la comparación

- **Más penalizadas por Evo2:** el consenso pone primero a VAR007, seguida de VAR014, VAR005 y VAR003. Cuatro de las cinco variantes del Top 5 inicial que Evo2 pudo evaluar (VAR007, VAR005, VAR003 y VAR011) quedan entre las cinco más penalizadas: hay **coincidencia** entre la evidencia genómica y Evo2.
- **Sube con Evo2:** VAR014 (p.Glu76Asp) pasa del lugar 7 al 2. Evo2 la penaliza de forma parecida en ambas hebras (−5.94 y −5.85 en el sitio) aunque el cambio químico es conservador (BLOSUM62 +2). Una explicación posible es que Evo2 mide qué tan esperada es la secuencia de ADN, no la química del aminoácido.
- **Baja con Evo2:** VAR003 (p.Arg408Trp) pasa del lugar 1 al 4: es la más negativa del panel en la ventana del gen (−31.65), pero casi nula en el sitio de esa hebra (+0.58). VAR011 pasa del 4 al 5 y también depende de la hebra (+0.15 en el genoma y −5.18 en el gen).
- **Sin dato de Evo2:** VAR012 (deleción) no se evaluó; su prioridad se sostiene solo con la evidencia genómica.
- Las discrepancias no se tratan como errores de Evo2 ni del ranking inicial: cada capa mide algo distinto.


## 7. Candidatos para RF3 y tabla de entrada

RF3 modela proteínas, así que solo sirven cambios de aminoácido con el formato `p.Arg408Trp`. Se eligen **dos variantes con evidencia convergente** (prioridad inicial alta y fuerte penalización en Evo2) para preguntar si el cambio altera el entorno estructural del residuo:

- **Candidato 1 · VAR007 p.Gly247Asp:** la más penalizada por Evo2 en el consenso de las cuatro medidas; cambia una glicina por un residuo con carga negativa en el dominio catalítico.
- **Candidato 2 · VAR003 p.Arg408Trp:** primer lugar del ranking inicial (BLOSUM62 −3, pierde una carga positiva) y la más penalizada del panel en la ventana del gen (−31.65).

La tabla `resultados/PAH_pre_vs_Evo2.tsv` tiene las columnas `ID` y `protein_change` que pide `resultados/rf3/Entradas_RF3.ipynb`.


In [6]:
CANDIDATOS_RF3 = ["VAR007", "VAR003"]

comp["seleccion_RF3"] = comp["ID"].isin(CANDIDATOS_RF3)
salida = comp[["ID", "protein_change", "transcript_change", "consequence", "dominio", "POPAF",
               "BLOSUM62", "puntaje_inicial", "rank_inicial"] + medidas +
              ["rank_evo2", "cambio_vs_inicial", "seleccion_RF3"]].sort_values("rank_inicial")
salida.to_csv("resultados/PAH_pre_vs_Evo2.tsv", sep="\t", index=False)
print(salida[salida["seleccion_RF3"]].to_string(index=False))


    ID protein_change transcript_change consequence    dominio   POPAF  BLOSUM62  puntaje_inicial  rank_inicial  delta_sitio_genoma  delta_ventana_genoma  delta_sitio_gen  delta_ventana_gen  rank_evo2  cambio_vs_inicial  seleccion_RF3
VAR003    p.Arg408Trp         c.1222C>T    missense catalítico 0.00092      -3.0              8.0             1               -5.72                -16.06             0.58             -31.65        4.0               -3.0           True
VAR007    p.Gly247Asp          c.740G>A    missense catalítico 0.00001      -1.0              7.5             3               -7.83                -22.99            -1.48             -21.58        1.0                2.0           True


## 8. Actualizar la tabla maestra

Se llenan las columnas de `resultados/tabla_maestra.tsv` que ya tienen datos. `RF3_or_structure_result`, `final_priority` y `limitations` se completan en `06_integracion.ipynb`.


In [7]:
tm = pd.read_csv("resultados/tabla_maestra.tsv", sep="\t", dtype=str)
info = comp.set_index("ID")

def texto_evo2(r):
    if pd.isna(r["delta_sitio_gen"]):
        return "No evaluada (deleción)"
    return (f"ΔlogP sitio gen {r['delta_sitio_gen']:.2f}; ventana gen {r['delta_ventana_gen']:.2f}; "
            f"lugar Evo2 {int(r['rank_evo2'])}/14")

for i, vid in enumerate(tm["variant_id"]):
    r = info.loc[vid]
    tm.loc[i, "genomic_location"] = r["genomic_location"]
    tm.loc[i, "region"] = f"{r['ubicacion']} ({r['dominio']})" if r["dominio"] != "fuera del CDS" else r["ubicacion"]
    tm.loc[i, "variant_type"] = r["variant_type"]
    tm.loc[i, "transcript_change"] = r["transcript_change"] if pd.notna(r["transcript_change"]) else ""
    tm.loc[i, "protein_change"] = r["protein_change"] if pd.notna(r["protein_change"]) else ""
    tm.loc[i, "initial_priority"] = str(int(r["rank_inicial"]))
    tm.loc[i, "evidence"] = f"{NOMBRE_CONSEC.get(r['consequence'], r['consequence'])}; " + evidencia(r)
    tm.loc[i, "Evo2_result"] = texto_evo2(r)

tm.to_csv("resultados/tabla_maestra.tsv", sep="\t", index=False)
print(tm[["variant_id", "region", "protein_change", "initial_priority", "Evo2_result"]].to_string(index=False))


variant_id           region     protein_change initial_priority                                                Evo2_result
    VAR001 CDS (catalítico)          p.Asn426=               12 ΔlogP sitio gen -0.45; ventana gen -1.11; lugar Evo2 10/14
    VAR002 CDS (catalítico)          p.Tyr414=               10  ΔlogP sitio gen -0.10; ventana gen 2.45; lugar Evo2 11/14
    VAR003 CDS (catalítico)        p.Arg408Trp                1  ΔlogP sitio gen 0.58; ventana gen -31.65; lugar Evo2 4/14
    VAR004 CDS (catalítico)          p.Leu385=               14   ΔlogP sitio gen 1.15; ventana gen 3.64; lugar Evo2 12/14
    VAR005 CDS (catalítico)        p.Asp338Ala                2 ΔlogP sitio gen -0.20; ventana gen -29.51; lugar Evo2 3/14
    VAR006 CDS (catalítico)          p.Leu321=               11   ΔlogP sitio gen 1.47; ventana gen 4.21; lugar Evo2 14/14
    VAR007 CDS (catalítico)        p.Gly247Asp                3 ΔlogP sitio gen -1.48; ventana gen -21.58; lugar Evo2 1/14
    VAR008 CDS (

## Cierre de la sección

- **Observamos:** el Top 5 inicial (antes de Evo2) y el orden de Evo2 coinciden en VAR007, VAR005, VAR003 y VAR011.
- **Inferimos que:** estas variantes tienen la evidencia de secuencia más consistente; VAR014 muestra una discrepancia entre química del cambio y Evo2.
- **La evidencia aporta:** anotación genómica, consecuencia, frecuencia, BLOSUM62, dominio y ΔlogP de Evo2.
- **Concluimos:** VAR007 y VAR003 pasan a RF3 como candidatos con evidencia convergente.
- **Falta evidencia:** efecto estructural (RF3, 6HYC) y cualquier medida experimental de actividad.
